# V-Mode Normalization Study (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** In Progress  
**Keywords:** Rubin Observatory, AOS, normalization, V-modes, sensitivity matrix  

## Description

Focused study of how different normalization schemes affect V-mode decomposition,
using a minimal 2-DOF, 1-Zernike example to make the algebra transparent.

The goal is to understand what normalization puts all DOFs on the same footing
in the normalized sensitivity matrix, so that V-modes properly combine degenerate DOFs.

**Output:** Printed tables of sensitivity matrix values, normalization weights, and V-modes

**Based on:** smatrix_vmode_info.ipynb

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |

## Index Notation Conventions

| Symbol | Meaning | Range |
|--------|---------|-------|
| $i$ | Degree of Freedom (DOF) index | 0-49 (or active subset) |
| $j$ | Pupil-plane Zernike index (Noll) | 4-26 |
| $k$ | Focal-plane Zernike index (double-Zernike $k_{\text{fp}}$) | 1, 2, 3 |
| $m$ | V-mode index | 1-$N_{\text{trunc}}$ |

## Table of Contents

1. [Setup & Imports](#setup)
2. [Parameters](#params)
3. [Build Sensitivity Matrix](#smatrix)
4. [Normalization Scheme 1: Default (r_i x f_i)](#norm1)
5. [Normalization Scheme 2: FWHM per Normalized DOF (1/f_i)](#norm2)
6. [Normalization Scheme 3: Geometric Mean sqrt(r_i / f_i)](#norm3)
7. [SVD and V-Mode Comparison](#svd)
8. [Analysis: Unit Invariance and DOF Equalization](#analysis)


<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.ts.ofc import OFCData, SensitivityMatrix, StateEstimator
from lsst.ts.ofc import BendModeToForce
from lsst.ts.wep.utils import convertZernikesToPsfWidth

%matplotlib inline

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# OFC config
ofc_config_dir = '/home/r/roodman/u/LSST/packages/ts_config_mttcs/MTAOS/v13/ofc'

# Zernike selection (pupil-plane, Noll indices)
zn_select = np.array([4])   # Just focus (Z4) for simplicity

# DOF selection (indices into the 50-DOF vector)
# Default: Cam_dz (index 5) and Cam_rx (index 8)
dof_names = ['Cam_dz', 'Cam_rx']
dof_indices = [5, 8]

# DOF labels (all 50)
labels_50dof = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]

# WFS corners
sensor_name_list = ['R00_SW0', 'R04_SW0', 'R40_SW0', 'R44_SW0']
n_wfs = len(sensor_name_list)

print(f'Zernikes: {zn_select}')
print(f'DOFs: {dof_names} (indices {dof_indices})')
print(f'WFS: {sensor_name_list}')
print(f'Measurement vector: {len(zn_select)} Zernike x {n_wfs} WFS = {len(zn_select) * n_wfs} elements')
print(f'DOFs: {len(dof_indices)}')

<a id='smatrix'></a>
## 1. Build Sensitivity Matrix

The raw sensitivity matrix $A_{ji}$ gives the Zernike wavefront change (in microns)
at each WFS position for a unit change in each DOF (microns for translations,
degrees for rotations, microns of peak motion for bending modes).

Our measurement vector is ordered as: [WFS0_z4, WFS1_z4, WFS2_z4, WFS3_z4]
(one Zernike at 4 field positions = 4 measurements, 2 DOFs).

In [ ]:
# Load OFCData and build sensitivity matrix
ofc_data = OFCData('lsst', config_dir=ofc_config_dir)
ofc_data.zn_selected = zn_select

field_angles = [ofc_data.sample_points[sensor] for sensor in sensor_name_list]
print('Field angles (deg):')
for s, fa in zip(sensor_name_list, field_angles):
    print(f'  {s}: ({fa[0]:.4f}, {fa[1]:.4f})')

# Evaluate sensitivity matrix: shape (n_wfs, n_zernike_all, n_dof_all)
dz_sens = SensitivityMatrix(ofc_data)
sens_3d = dz_sens.evaluate(field_angles, rotation_angle=0.0)

# Select our Zernikes and reshape to 2D
zn_idx = zn_select - 4  # Noll -> 0-based
sens_sub = sens_3d[:, zn_idx, :]  # (n_wfs, n_zk, 50)
n_zk = len(zn_select)
n_meas = n_wfs * n_zk

# Reshape: measurement vector = (WFS0_z4, WFS1_z4, ...) x DOFs
A_full = sens_sub.reshape((n_meas, -1))  # (4, 50)

# Select our DOFs
A = A_full[:, dof_indices]  # (4, 2)

print(f'\nRaw sensitivity matrix A (shape {A.shape}):')
print(f'  Rows: {n_meas} measurements ({n_wfs} WFS x {n_zk} Zernike)')
print(f'  Cols: {len(dof_indices)} DOFs')

print(f'\n{"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    zk_idx = row_idx % n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[zk_idx]}'
    print(f'{label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A[row_idx, col_idx]:>14.6e}', end='')
    print()

print(f'\nUnits: microns of wavefront per unit DOF')
print(f'  Cam_dz unit: microns')
print(f'  Cam_rx unit: degrees')

### 1.1 Normalization Weight Components

The normalization has two factors per DOF:
- $r_i$: **range weight** — physical stroke range (microns or arcsec) divided by max actuator force
- $f_i$: **FWHM weight** — RSS of FWHM sensitivity across all field positions

The default OFC normalization is $n_i = r_i \times f_i$.

In [ ]:
# Compute range weights r_i
m1m3_bending_range = ofc_data.m1m3_force_range / 20
m2_bending_range = ofc_data.m2_force_range / 20
m1m3_bmf = BendModeToForce('M1M3', ofc_data)
m2_bmf = BendModeToForce('M2', ofc_data)

range_weights_50 = np.concatenate((
    ofc_data.rb_stroke,
    m1m3_bending_range / np.max(np.abs(m1m3_bmf.rot_mat), axis=0),
    m2_bending_range / np.max(np.abs(m2_bmf.rot_mat), axis=0),
))

# Compute FWHM weights f_i
sensitivity_matrix = dz_sens.evaluate(field_angles, rotation_angle=0.0)
sensitivity_matrix = sensitivity_matrix[:, ofc_data.zn_idx, :]
fwhm_matrix = np.zeros(sensitivity_matrix.shape)
for idy in range(sensitivity_matrix.shape[0]):
    fwhm_matrix[idy, ...] = convertZernikesToPsfWidth(sensitivity_matrix[idy, ...].T).T
fwhm_matrix_2d = fwhm_matrix.reshape((-1, fwhm_matrix.shape[2]))
fwhm_weights_50 = np.zeros(50)
for i in range(50):
    fwhm_weights_50[i] = np.sqrt(np.sum(np.square(fwhm_matrix_2d[:, i])))

# Extract for our DOFs
r_i = range_weights_50[dof_indices]
f_i = fwhm_weights_50[dof_indices]
n_default = ofc_data.normalization_weights[dof_indices]

print('Normalization weight components for selected DOFs:')
print(f'{"DOF":>10s} {"r_i (range)":>14s} {"f_i (FWHM)":>14s} {"r_i * f_i":>14s} {"n_i (stored)":>14s}')
print('-' * 66)
for idx, name in enumerate(dof_names):
    print(f'{name:>10s} {r_i[idx]:>14.6e} {f_i[idx]:>14.6e} {r_i[idx]*f_i[idx]:>14.6e} {n_default[idx]:>14.6e}')

print(f'\nPhysical interpretation:')
for idx, name in enumerate(dof_names):
    print(f'  {name}: range = {r_i[idx]:.4f} (physical stroke), '
          f'FWHM sensitivity = {f_i[idx]:.4f} arcsec FWHM/unit DOF')

# Derive stored f_i from stored normalization weights and computed r_i
f_i_stored = n_default / r_i

print(f'\nStored vs computed FWHM weights:')
print(f'{"DOF":>10s} {"f_i (computed)":>16s} {"f_i (stored)":>16s} {"ratio":>10s}')
print('-' * 56)
for idx, name in enumerate(dof_names):
    ratio = f_i[idx] / f_i_stored[idx] if f_i_stored[idx] != 0 else float('inf')
    print(f'{name:>10s} {f_i[idx]:>16.6e} {f_i_stored[idx]:>16.6e} {ratio:>10.4f}')


<a id='norm1'></a>
## 2. Normalization Scheme 1: Default ($\tilde{A} = A \cdot \text{diag}(r_i \times f_i)$)

We show two variants: (a) using stored normalization weights, and (b) using our computed $r_i \times f_i$.
The $r_i$ values match, but the $f_i$ values may differ.

The current OFC default multiplies each column of $A$ by $n_i = r_i \times f_i$.
This converts the sensitivity matrix from "microns of wavefront per physical DOF unit"
to "microns of wavefront per dimensionless DOF unit", where the dimensionless unit
combines range and FWHM sensitivity.

**Caveat — not unit-invariant.** Under a change of physical units for DOF $i$
($x_i \to \alpha x_i$), the column $A_{:,i}$ scales as $\alpha$, $r_i$ as $1/\alpha$,
and $f_i$ as $\alpha$. The product $r_i \times f_i$ is dimensionless ($\propto \alpha^0$),
so $\tilde A_{:,i} = A_{:,i} \cdot n_i$ scales as $\alpha$ — i.e. the *normalized*
sensitivity matrix changes if you re-express a DOF in different units. This is undesirable.
Schemes 2 and 3 below are unit-invariant.


In [ ]:
# Scheme 1a: stored normalization weights
n1a = n_default.copy()
A1a = A @ np.diag(n1a)

# Scheme 1b: computed r_i * f_i
n1b = r_i * f_i
A1b = A @ np.diag(n1b)

print('Scheme 1a: Stored normalization weights')
print('=' * 70)
print(f'\nNormalization weights (stored):')
for idx, name in enumerate(dof_names):
    print(f'  n_i({name}) = {n1a[idx]:.6e}  (= r_i * f_i_stored = {r_i[idx]:.4e} * {f_i_stored[idx]:.4e})')

print(f'\nNormalized A_1a = A @ diag(n_stored):')
print(f'{"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'{label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A1a[row_idx, col_idx]:>14.6e}', end='')
    print()

col_norms_A1a = np.linalg.norm(A1a, axis=0)
print(f'\nColumn norms: {dof_names[0]} = {col_norms_A1a[0]:.6e}, '
      f'{dof_names[1]} = {col_norms_A1a[1]:.6e}, '
      f'ratio = {col_norms_A1a[0]/col_norms_A1a[1]:.4f}')

print(f'\n\nScheme 1b: Computed r_i * f_i')
print('=' * 70)
print(f'\nNormalization weights (computed):')
for idx, name in enumerate(dof_names):
    print(f'  n_i({name}) = r_i * f_i = {r_i[idx]:.4e} * {f_i[idx]:.4e} = {n1b[idx]:.6e}')

print(f'\nNormalized A_1b = A @ diag(r_i * f_i_computed):')
print(f'{"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'{label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A1b[row_idx, col_idx]:>14.6e}', end='')
    print()

col_norms_A1b = np.linalg.norm(A1b, axis=0)
print(f'\nColumn norms: {dof_names[0]} = {col_norms_A1b[0]:.6e}, '
      f'{dof_names[1]} = {col_norms_A1b[1]:.6e}, '
      f'ratio = {col_norms_A1b[0]/col_norms_A1b[1]:.4f}')

col_norms_A = np.linalg.norm(A, axis=0)
print(f'\n\nColumn norm summary:')
print(f'  Raw A ratio (Cam_dz/Cam_rx): {col_norms_A[0]/col_norms_A[1]:.4f}')
print(f'  Scheme 1a (stored) ratio:    {col_norms_A1a[0]/col_norms_A1a[1]:.4f}')
print(f'  Scheme 1b (computed) ratio:  {col_norms_A1b[0]/col_norms_A1b[1]:.4f}')

<a id='norm2'></a>
## 3. Normalization Scheme 2: FWHM per Normalized DOF

This scheme proceeds in steps:

**Step 1:** Define a "Normalized DOF" (NDOF): $\text{NDOF}_i = \text{DOF}_i / r_i$

This makes all DOFs dimensionless and scaled to their physical range.
The sensitivity matrix becomes $A \cdot \text{diag}(r_i)$: microns of wavefront per unit NDOF.

**Step 2:** Compute FWHM sensitivity per unit NDOF: $\hat{f}_i = f_i \times r_i$

This is the FWHM change (arcsec) for a unit change in NDOF.

**Step 3:** Normalize the sensitivity matrix by dividing each column by $\hat{f}_i$:

$$\tilde{A}_2 = A \cdot \text{diag}(r_i) \cdot \text{diag}(1/\hat{f}_i) = A \cdot \text{diag}(r_i / (f_i \cdot r_i)) = A \cdot \text{diag}(1/f_i)$$

The result has units: microns of wavefront per arcsec of FWHM change.
This is the `inv_f` normalization mode in smatrix_vmode_info.

In [ ]:
# Scheme 2: FWHM per Normalized DOF
print('Scheme 2: FWHM per Normalized DOF')
print('=' * 70)

# Step 1: Sensitivity per Normalized DOF
A_ndof = A @ np.diag(r_i)
print(f'\nStep 1: A_ndof = A @ diag(r_i)  [microns wavefront / unit NDOF]')
print(f'  r_i values:')
for idx, name in enumerate(dof_names):
    print(f'    r_i({name}) = {r_i[idx]:.6e}')

print(f'\n  A_ndof:')
print(f'  {"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'  {label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A_ndof[row_idx, col_idx]:>14.6e}', end='')
    print()

# Step 2: FWHM sensitivity per unit NDOF
f_hat = f_i * r_i
print(f'\nStep 2: Normalized FWHM weight f_hat_i = f_i * r_i  [arcsec FWHM / unit NDOF]')
for idx, name in enumerate(dof_names):
    print(f'  f_hat({name}) = {f_i[idx]:.6e} * {r_i[idx]:.6e} = {f_hat[idx]:.6e}')

# Step 3: Normalize by FWHM impact
n2 = 1.0 / f_i  # = r_i / f_hat = r_i / (f_i * r_i) = 1/f_i
A2 = A @ np.diag(n2)

print(f'\nStep 3: Ã₂ = A_ndof @ diag(1/f_hat) = A @ diag(1/f_i)')
print(f'  [microns wavefront / arcsec FWHM change]')
print(f'  n_i = 1/f_i:')
for idx, name in enumerate(dof_names):
    print(f'    n_i({name}) = 1/{f_i[idx]:.6e} = {n2[idx]:.6e}')

print(f'\n  Ã₂:')
print(f'  {"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'  {label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A2[row_idx, col_idx]:>14.6e}', end='')
    print()

col_norms_A2 = np.linalg.norm(A2, axis=0)
print(f'\nColumn norms (||Ã₂_i||):')
for idx, name in enumerate(dof_names):
    print(f'  {name}: {col_norms_A2[idx]:.6e}')
print(f'  Ratio (Cam_dz/Cam_rx): {col_norms_A2[0]/col_norms_A2[1]:.4f}')

<a id='norm3'></a>
## 4. Normalization Scheme 3: Geometric Mean ($n_i = \sqrt{r_i / f_i}$)

A unit-invariant scheme that combines **both** range and optical impact.

**Construction.** Both $r_i$ and $1/f_i$ scale as $1/\alpha$ under unit changes
($x_i \to \alpha x_i$), so any combination $n_i \propto r_i^{a} f_i^{-(1-a)}$
with $a \in [0,1]$ keeps $\tilde A$ unit-invariant. Choosing $a = 1/2$ gives the
symmetric geometric mean

$$ n_i = \sqrt{\frac{r_i}{f_i}}, \qquad q_i = x_i \sqrt{\frac{f_i}{r_i}} $$

**Interpretation.** Squaring,

$$ q_i^2 = \underbrace{(x_i f_i)}_{\text{FWHM contribution (arcsec)}} \,\cdot\, \underbrace{(x_i / r_i)}_{\text{fraction of range}} $$

is the geometric mean of the optical cost and the mechanical cost of moving DOF $i$
to position $x_i$. The SVD truncated tail then preferentially drops modes that are
**both** optically weak **and** range-hungry — exactly the modes you want to ignore.

**Limits:** $a=0 \to 1/f$ (Scheme 2, pure optical equalization);
$a=1 \to r$ (pure range protection); $a=1/2 \to $ symmetric blend.


In [ ]:
# Scheme 3: geometric mean n_i = sqrt(r_i / f_i)  (unit-invariant)
n3 = np.sqrt(r_i / f_i)
A3 = A @ np.diag(n3)

print('Scheme 3: Geometric Mean n_i = sqrt(r_i / f_i)')
print('=' * 70)
print(f'\nNormalization weights:')
for idx, name in enumerate(dof_names):
    print(f'  n_i({name}) = sqrt({r_i[idx]:.4e} / {f_i[idx]:.4e}) = {n3[idx]:.6e}')

print(f'\nDimensionless DOF q_i = x_i * sqrt(f_i / r_i):')
for idx, name in enumerate(dof_names):
    print(f'  q_i({name}) = x_i * {np.sqrt(f_i[idx]/r_i[idx]):.6e}')

print(f'\nNormalized A_3 = A @ diag(sqrt(r_i / f_i)):')
print(f'  {"":>10s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print()
for row_idx in range(n_meas):
    wfs_idx = row_idx // n_zk
    label = f'{sensor_name_list[wfs_idx]}_z{zn_select[0]}'
    print(f'  {label:>10s}', end='')
    for col_idx in range(len(dof_indices)):
        print(f'{A3[row_idx, col_idx]:>14.6e}', end='')
    print()

col_norms_A3 = np.linalg.norm(A3, axis=0)
print(f'\nColumn norms (||A_3_i||):')
for idx, name in enumerate(dof_names):
    print(f'  {name}: {col_norms_A3[idx]:.6e}')
print(f'  Ratio (Cam_dz/Cam_rx): {col_norms_A3[0]/col_norms_A3[1]:.4f}')

# Tunable variant: n_i = r_i^a * f_i^(a-1)
print(f'\n\nTunable family n_i = r_i^a * f_i^(a-1), a in [0,1]:')
print(f'{"a":>6s} {"n_i(Cam_dz)":>16s} {"n_i(Cam_rx)":>16s} {"col_norm_ratio":>16s}')
print('-' * 56)
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    n_a = r_i**a * f_i**(a - 1)
    A_a = A @ np.diag(n_a)
    cn = np.linalg.norm(A_a, axis=0)
    print(f'{a:>6.2f} {n_a[0]:>16.6e} {n_a[1]:>16.6e} {cn[0]/cn[1]:>16.4f}')


<a id='svd'></a>
## 5. SVD and V-Mode Comparison

Compare the SVD of the two normalized sensitivity matrices.
In a well-normalized system where two DOFs have equal "importance",
the V-modes should mix them based on their physical degeneracy,
not just because one has a larger column norm.

In [ ]:
# SVD of all schemes
U0, s0, Vh0 = np.linalg.svd(A, full_matrices=False)
V0 = Vh0.T

U1a, s1a, Vh1a = np.linalg.svd(A1a, full_matrices=False)
V1a = Vh1a.T

U1b, s1b, Vh1b = np.linalg.svd(A1b, full_matrices=False)
V1b = Vh1b.T

U2, s2, Vh2 = np.linalg.svd(A2, full_matrices=False)
V2 = Vh2.T

U3, s3, Vh3 = np.linalg.svd(A3, full_matrices=False)
V3 = Vh3.T

print('SVD Comparison')
print('=' * 70)

for label, sv, V in [('Raw A (no normalization)', s0, V0),
                      ('Scheme 1a (stored r*f, NOT unit-invariant)', s1a, V1a),
                      ('Scheme 1b (computed r*f, NOT unit-invariant)', s1b, V1b),
                      ('Scheme 2 (1/f, unit-invariant)', s2, V2),
                      ('Scheme 3 (sqrt(r/f), unit-invariant)', s3, V3)]:
    print(f'\n{label}:')
    print(f'  Singular values: σ = [{", ".join(f"{v:.6f}" for v in sv)}]')
    print(f'  Condition number: {sv[0]/sv[-1]:.4f}')
    print(f'  V matrix (rows = DOFs, cols = V-modes):')
    for idx, name in enumerate(dof_names):
        print(f'    {name:>10s}: [{", ".join(f"{V[idx, m]:>+8.5f}" for m in range(len(sv)))}]')


In [ ]:
# Visual comparison of V-modes
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

schemes = [
    ('Raw A', V0, s0),
    ('1a: stored r*f', V1a, s1a),
    ('1b: computed r*f', V1b, s1b),
    ('2: 1/f', V2, s2),
    ('3: sqrt(r/f)', V3, s3),
]

for ax, (title, V, sv) in zip(axes, schemes):
    im = ax.imshow(V, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(sv)), [f'm={m+1}\nσ={sv[m]:.3f}' for m in range(len(sv))], fontsize=8)
    ax.set_yticks(range(len(dof_names)), dof_names)
    ax.set_xlabel('V-mode')
    ax.set_ylabel('DOF')
    ax.set_title(title, fontsize=10)
    for i in range(V.shape[0]):
        for m in range(V.shape[1]):
            ax.text(m, i, f'{V[i,m]:+.3f}', ha='center', va='center', fontsize=10,
                    color='white' if abs(V[i,m]) > 0.5 else 'black')

fig.suptitle('V-Mode Composition Under Different Normalizations', fontsize=13)
fig.tight_layout()
plt.show()


<a id='analysis'></a>
## 6. Analysis: Unit Invariance and DOF Equalization

Two criteria for a "good" normalization:

**1. Unit invariance of $\tilde A$.** Under $x_i \to \alpha x_i$ (e.g. switching µm $\to$ mm),
the column $A_{:,i}$ scales as $\alpha$. For $\tilde A_{:,i} = A_{:,i} n_i$ to be invariant,
$n_i$ must scale as $1/\alpha$. The scaling table is:

| Scheme | $n_i$ scaling | $\tilde A$ invariant? |
|---|---|---|
| $r_i$ | $1/\alpha$ | yes |
| $1/f_i$ | $1/\alpha$ | yes |
| $r_i \cdot f_i$ | $1$ | **no** ($\tilde A \propto \alpha$) |
| $\sqrt{r_i / f_i}$ | $1/\alpha$ | yes |

Only schemes whose factors collectively scale as $1/\alpha$ produce a normalized
sensitivity matrix that is independent of how the user chose to express each DOF.

**2. DOF equalization.** Equal column norms $\|\tilde A_{:,i}\|$ mean equal "weight"
in the SVD — degenerate DOFs combine on physical, not numerical, grounds.

Below we tabulate column-norm ratios and the physical meaning of "1 unit of normalized DOF"
for each scheme.


In [ ]:
print('Analysis: Column Norms and DOF Equalization')
print('=' * 70)

print(f'\nColumn norms ||A_i|| for each normalization:')
print(f'{"Scheme":>30s}', end='')
for name in dof_names:
    print(f'{name:>14s}', end='')
print(f'{"ratio":>10s}')
print('-' * 68)

for scheme_name, A_norm in [('Raw A', A),
                             ('1a: stored r*f', A1a),
                             ('1b: computed r*f', A1b),
                             ('2: 1/f (unit-invariant)', A2),
                             ('3: sqrt(r/f) (unit-invariant)', A3)]:
    norms = np.linalg.norm(A_norm, axis=0)
    ratio = norms[0] / norms[1]
    print(f'{scheme_name:>30s}', end='')
    for n in norms:
        print(f'{n:>14.6e}', end='')
    print(f'{ratio:>10.4f}')

print(f'\n\nPhysical meaning of "1 unit of normalized DOF":')

print(f'\nScheme 1a (stored r*f): 1 unit of dimensionless DOF corresponds to:')
for idx, name in enumerate(dof_names):
    n_val = n1a[idx]
    fwhm_val = f_i_stored[idx] * n_val
    print(f'  {name}: {n_val:.4e} physical units -> {fwhm_val:.4e} arcsec FWHM')

print(f'\nScheme 2 (1/f): 1 unit -> exactly 1 arcsec FWHM for every DOF')
print(f'Scheme 3 (sqrt(r/f)): 1 unit -> sqrt(r_i * f_i) arcsec FWHM (= geom mean of FWHM and range cost)')
for idx, name in enumerate(dof_names):
    print(f'  {name}: 1 unit = {n3[idx]:.4e} physical units, '
          f'= {f_i[idx]*n3[idx]:.4e} arcsec FWHM, '
          f'= {n3[idx]/r_i[idx]:.4e} of range')

# --- Numerical demonstration of unit invariance ---
print(f'\n\nUnit-invariance demonstration: re-express Cam_dz in mm instead of microns')
print('(alpha = 1000; A column scales by 1000, r_i and 1/f_i scale by 1/1000)')
print('-' * 68)
alpha = 1000.0
A_alt = A.copy()
A_alt[:, 0] *= alpha   # column scales by alpha when DOF unit gets alpha times bigger
r_alt = r_i.copy();   r_alt[0] /= alpha
f_alt = f_i.copy();   f_alt[0] *= alpha

n1b_alt = r_alt * f_alt
n2_alt  = 1.0 / f_alt
n3_alt  = np.sqrt(r_alt / f_alt)

A1b_alt = A_alt @ np.diag(n1b_alt)
A2_alt  = A_alt @ np.diag(n2_alt)
A3_alt  = A_alt @ np.diag(n3_alt)

def maxdiff(M1, M2):
    return np.max(np.abs(M1 - M2))

print(f'  max|A_1b(mm) - A_1b(um)|  = {maxdiff(A1b_alt, A1b):.4e}   '
      f'(scheme 1b: {"INVARIANT" if maxdiff(A1b_alt, A1b) < 1e-10 else "NOT invariant"})')
print(f'  max|A_2(mm)  - A_2(um)|   = {maxdiff(A2_alt,  A2 ):.4e}   '
      f'(scheme 2:  {"INVARIANT" if maxdiff(A2_alt,  A2 ) < 1e-10 else "NOT invariant"})')
print(f'  max|A_3(mm)  - A_3(um)|   = {maxdiff(A3_alt,  A3 ):.4e}   '
      f'(scheme 3:  {"INVARIANT" if maxdiff(A3_alt,  A3 ) < 1e-10 else "NOT invariant"})')

print()
print('=' * 70)
print('CONCLUSION: Normalization for Rubin Active Optics')
print('=' * 70)
print("""
Two requirements:
  (i)  Tilde-A must be invariant under choice of physical units for each DOF.
  (ii) The normalization should reflect both image-quality impact (f_i) AND
       hardware range (r_i), since both are physical constraints we must obey.

The unit-invariant family n_i = r_i^a * f_i^(a-1), a in [0,1], satisfies (i)
for any a. The endpoints are:

  a = 0 -> n_i = 1/f_i  (Scheme 2, pure optical equalization; ignores range)
  a = 1 -> n_i = r_i    (pure range protection; ignores optics)

Scheme 1 (n_i = r_i * f_i) FAILS criterion (i): tilde-A scales as alpha when
units change, so the SVD result is an artifact of the unit choice.

RECOMMENDATION: Scheme 3, n_i = sqrt(r_i / f_i)  (a = 1/2 geometric mean).
  - Unit-invariant tilde-A (criterion i).
  - Symmetric weighting of range and optical cost (criterion ii).
  - q_i^2 = (x_i * f_i) * (x_i / r_i) is the geometric mean of FWHM
    contribution (arcsec) and range fraction.  The SVD's truncated tail
    then preferentially discards modes that are both optically weak AND
    range-hungry -- exactly what we want to drop.
  - The tunable exponent a lets us slide between range-priority and
    optical-priority while keeping unit invariance.
""")
